In [1]:
import DataGen as dtg
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from tqdm.auto import tqdm
from efficient_kan.kan import KAN
from torch.utils.data import TensorDataset, DataLoader, Sampler

import torch.optim as optim

SEED = 42

c:\Users\Conciso TI\.conda\envs\exai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using '{DEVICE}' device.")

Using 'cpu' device.


## Obtenção do conjunto de dados

### Definindo o espaço de velocidade


In [3]:
X_POINTS = 30
Z_POINTS = 30
Y_POINTS = 1

X_DISTANCE = 1
Y_DISTANCE = 1
Z_DISTANCE = 1


In [4]:
dg_over = 10 / 100
xover = np.arange(0, 2900 / 1000 + dg_over, dg_over)
zover = np.arange(0, 2900 / 1000 + dg_over, dg_over)

v_line_z = 1.5 + zover
v_line_x = np.tile(0.3 * xover, (len(zover), 1))
v_depth = np.tile(v_line_z[:, np.newaxis], (1, len(xover))) + v_line_x

v_depth = v_depth.reshape(X_POINTS,Z_POINTS,Y_POINTS) # necessário adicionar a terceira dimensão para o modelo do Solver

In [5]:
df = dtg.generateMultipleData(
    nPoints = (X_POINTS, Z_POINTS, Y_POINTS), 
    nodeDistances = (X_DISTANCE, Y_DISTANCE, Z_DISTANCE),
    velocityModel = v_depth,
    sourceOriginInterval= [[0,0],[2,2]])

In [6]:
df.head(5)

,x,z,travel_time,source_x0,source_z0
0,0,0,0.000000,0,0
1,0,1,0.653595,0,0
2,0,2,1.298810,0,0
3,0,3,1.933169,0,0
4,0,4,2.556145,0,0


In [7]:
features = ['x', 'z', 'source_x0', 'source_z0']
target = ['travel_time']

### Separando o conjunto em Treinamento, Validação e Teste

In [8]:
initial_conditions = df[['source_x0', 'source_z0']].drop_duplicates()
aux = initial_conditions.copy()

ic_train = aux.sample(frac=0.6, random_state=SEED)
ic_val = aux.drop(ic_train.index).sample(frac=0.5, random_state=SEED)
ic_test = aux.drop(list(ic_train.index) + list(ic_val.index))

print("Train Initial Conditions:", len(ic_train))
print("Validation Initial Conditions:", len(ic_val))
print("Test Initial Conditions:", len(ic_test))

Train Initial Conditions: 5
Validation Initial Conditions: 0
Test Initial Conditions: 0
